In [215]:
import grewpy
import yaml
import sys
import numpy as np

sys.path.insert(1, '/Users/madalina/Documents/M2TAL/stage/grex/grex2')
import pyximport
pyximport.install()
import grex.data
import grex.utils
import grex.features

# path = "/Users/madalina/Documents/M1TAL/stage-SK/Treebanks/UD_French-GSD-master"
# grewpy.set_config('ud')
path = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/input/SUD_French-GSD-r2.15"
grewpy.set_config('sud')
corpus = grewpy.Corpus(path)
draft = grewpy.CorpusDraft(corpus)

In [216]:
all_matches = corpus.search(grewpy.Request("pattern{X[upos<>PUNCT]}").without("X[InIdiom=Yes]").without("X[Idiom=Yes]").without("X[InTitle=Yes]").without("X[Title=Yes]").without("X[Scrap=Yes]").without("X[Foreign]").without("X[Lang]").without("X-[fixed]->Y").without("Y-[flat:name]->X").without("Y-[goeswith]->X") , clustering_parameter=['X.lemma'])

In [220]:
print(len(all_matches["amateur"]))

11


In [195]:
matches = {}
for key, value in all_matches.items():
    # remove those that have less than 10 occurrences
    if len(value) > 10:
        matches[key] = value

In [196]:
print(len(matches))

2947


In [251]:
import json 
# Create a dictionary to map sent_id to sentences for quick lookup
sent_id_to_sentence = {draft[i].meta['sent_id']: draft[i].features for i in range(len(draft))}

match_upos = {}
for key, value in matches.items():
    for m in value:
        match_sent_id = m['sent_id']
        match_node_index = str(m['matching']['nodes']['X'])
        if match_sent_id in sent_id_to_sentence:
            current_sentence_features = sent_id_to_sentence[match_sent_id]
            if match_node_index in current_sentence_features.keys():
                current_token_features = current_sentence_features[match_node_index]
                if 'ExtPos' in current_token_features:
                    match_upos.setdefault((key, current_token_features['ExtPos']), []).append(m)
                else:
                    match_upos.setdefault((key, current_token_features['upos']), []).append(m)
                


# # write the dictionary to a json file
# with open('match_upos.json', 'w') as f:
#     json.dump(match_upos, f)

In [252]:
new_match_upos = {}
for key, value in match_upos.items():
    if len(value) > 10:
        new_match_upos[key] = value
match_upos = new_match_upos

for key, value in match_upos.items():
    print(key, len(value))

('€', 'NOUN') 45
('œuvrer', 'VERB') 12
('œuvre', 'NOUN') 115
('œuf', 'NOUN') 19
('œil', 'NOUN') 33
('île', 'NOUN') 124
('être', 'AUX') 9438
('être', 'VERB') 149
('évêque', 'NOUN') 45
('événement', 'NOUN') 50
('évènement', 'NOUN') 20
('évoquer', 'VERB') 29
('évolution', 'NOUN') 39
('évoluer', 'VERB') 65
('éviter', 'VERB') 42
('évidence', 'NOUN') 13
('éventuellement', 'ADV') 12
('éventuel', 'ADJ') 13
('évaluer', 'VERB') 11
('été', 'NOUN') 57
('étudier', 'VERB') 45
('étudiant', 'NOUN') 23
('étude', 'NOUN') 99
('étranger', 'ADJ') 45
('étranger', 'NOUN') 16
('étrange', 'ADJ') 20
('étoile', 'NOUN') 42
('étendre', 'VERB') 36
('état', 'NOUN') 149
('étape', 'NOUN') 28
('étang', 'NOUN') 11
('étage', 'NOUN') 20
('établissement', 'NOUN') 43
('établir', 'VERB') 75
('équiper', 'VERB') 22
('équipement', 'NOUN') 21
('équipe', 'NOUN') 233
('équipage', 'NOUN') 11
('épreuve', 'NOUN') 41
('épouser', 'VERB') 27
('épouse', 'NOUN') 32
('époque', 'NOUN') 100
('épisode', 'NOUN') 49
('énorme', 'ADJ') 11
('énerg

In [198]:
# upos_of_matches = {}
# for key, value in match_upos.items():
#     upos_of_matches[key] = list(set(value))

In [253]:
with open("../3. probability_matrix/patterns_all_nodes.txt") as instream:
    config = yaml.load(instream, Loader=yaml.Loader)

templates = grex.utils.FeaturePredicate.from_config(config["templates"])
feature_predicate = grex.utils.FeaturePredicate.from_config(config["features"], templates=templates)

In [254]:
data = { k : list() for k in match_upos }
for node, mts in match_upos.items():
    for match in mts:
        features = grex.data.extract_features(draft, match, feature_predicate)
        formatted_features = [
            f"{':'.join(k)}={v}" if not isinstance(v, set) else
            f"{':'.join(k)}={val}" for k, v in features.items() for val in (v if isinstance(v, set) else [v])
        ]
        data[node].append(formatted_features)

In [255]:
unique_lemma = sorted(set([k for k in data]))
unique_features = sorted(set([feat for _, match_upos in data.items() for m in match_upos for feat in m]))

idx2feature = {i : feat for i, feat in enumerate(unique_features) }
feature2idx = {feat : i for i, feat in idx2feature.items()}
idx2adv = {i : feat for i, feat in enumerate(unique_lemma) }
adv2idx = {feat : i for i, feat in idx2adv.items()}

In [256]:
unique_features

['node:X:child:Cxn=Conditional-NegativeEpistemic',
 'node:X:child:Cxn=Conditional-NegativeEpistemic,Interrogative-WHInfo-Direct',
 'node:X:child:Cxn=Conditional-NegativeEpistemic,Interrogative-WHInfo-Indirect',
 'node:X:child:Cxn=Conditional-NeutralEpistemic',
 'node:X:child:Cxn=Conditional-NeutralEpistemic,Conditional-Reduced',
 'node:X:child:Cxn=Conditional-NeutralEpistemic,Existential-HavePred-ItExpl-ThereExpl',
 'node:X:child:Cxn=Conditional-NeutralEpistemic,Interrogative-Polar-Direct',
 'node:X:child:Cxn=Conditional-NeutralEpistemic,Interrogative-WHInfo-Direct',
 'node:X:child:Cxn=Existential-HavePred-ItExpl-ThereExpl',
 'node:X:child:Cxn=Interrogative-Alternative',
 'node:X:child:Cxn=Interrogative-Alternative,Interrogative-WHInfo-Direct',
 'node:X:child:Cxn=Interrogative-Polar-Direct',
 'node:X:child:Cxn=Interrogative-Polar-Indirect',
 'node:X:child:Cxn=Interrogative-WHInfo-Direct',
 'node:X:child:Cxn=Interrogative-WHInfo-Indirect',
 'node:X:child:Cxn=NPN',
 'node:X:child:CxnElt=

In [257]:
X = np.zeros((len(data.keys()), len(unique_features)))
for adv, samples in data.items():
    n_samples = len(matches)
    for m in samples:
        for feature in m:
            X[adv2idx[adv], feature2idx[feature]] += 1
    X[adv2idx[adv]] = X[adv2idx[adv]] / n_samples
print(f"{X.shape=}")

X.shape=(2978, 2001)


In [258]:
import numpy as np
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import silhouette_score

def find_optimal_clusters(X, max_clusters=20, metric='cosine', method='complete'):
    distance_matrix = pdist(X, metric=metric)
    linked = linkage(distance_matrix, method=method, optimal_ordering=True)
    
    silhouette_scores = []
    for num_clusters in range(2, max_clusters + 1):
        labels = fcluster(linked, num_clusters, criterion='maxclust')
        if len(np.unique(labels)) > 1:  # Ensure there is more than one cluster
            score = silhouette_score(X, labels, metric=metric)
            silhouette_scores.append(score)
            print(f'Number of clusters: {num_clusters}, Silhouette Score: {score}')
        else:
            silhouette_scores.append(-1)  # Append a low score if only one cluster
    
    optimal_clusters = np.argmax(silhouette_scores) + 2  # +2 because range starts from 2
    return optimal_clusters, silhouette_scores

# Find the optimal number of clusters
optimal_clusters, silhouette_scores = find_optimal_clusters(X, max_clusters=50)
print(f'Optimal number of clusters: {optimal_clusters}')
print(X.shape)

Number of clusters: 2, Silhouette Score: 0.29567671218968417
Number of clusters: 3, Silhouette Score: 0.3397989505290858
Number of clusters: 4, Silhouette Score: 0.36153153902475266
Number of clusters: 5, Silhouette Score: 0.3699453332030302
Number of clusters: 6, Silhouette Score: 0.3654280756028161
Number of clusters: 7, Silhouette Score: 0.5001463953195933
Number of clusters: 8, Silhouette Score: 0.49362871370845696
Number of clusters: 9, Silhouette Score: 0.4815418358370537
Number of clusters: 10, Silhouette Score: 0.4660501459230713
Number of clusters: 11, Silhouette Score: 0.4638394050426276
Number of clusters: 12, Silhouette Score: 0.47708741105770197
Number of clusters: 13, Silhouette Score: 0.47898673754165244
Number of clusters: 14, Silhouette Score: 0.47362556065114725
Number of clusters: 15, Silhouette Score: 0.474526858849437
Number of clusters: 16, Silhouette Score: 0.5075973806433883
Number of clusters: 17, Silhouette Score: 0.5094939907541739
Number of clusters: 18, Sil

In [259]:
# see what there is in each cluster
distance_matrix = pdist(X, metric='cosine')
# Complete -> we cluster 
linked = linkage(distance_matrix, method="complete", optimal_ordering=True)
labels = fcluster(linked, optimal_clusters, criterion='maxclust')
clusters = {i: [] for i in range(1, optimal_clusters + 1)}
for i, label in enumerate(labels):
    clusters[label].append(i)

# for cluster, members in clusters.items():
#     print(f'Cluster {cluster}:')
#     for member in members:
#         print(f'  {unique_lemma[member]}')

# for member in clusters[1]:
#     print(f'  {unique_lemma[member]}')

In [264]:
for cluster, members in clusters.items():
    print(f'Cluster {cluster}:')
    for member in members:
        print(f'  {unique_lemma[member]}')

Cluster 1:
  ('absolument', 'ADV')
  ('actuellement', 'ADV')
  ('ailleurs', 'ADV')
  ('ainsi', 'ADV')
  ('alors', 'ADV')
  ('après', 'ADV')
  ('assez', 'ADV')
  ("aujourd'hui", 'ADV')
  ('auparavant', 'ADV')
  ('aussi', 'ADV')
  ('autant', 'ADV')
  ('autrement', 'ADV')
  ('beaucoup', 'ADV')
  ('bien', 'ADV')
  ('bientôt', 'ADV')
  ('cependant', 'ADV')
  ('certes', 'ADV')
  ('clairement', 'ADV')
  ('complètement', 'ADV')
  ('davantage', 'ADV')
  ('depuis', 'ADV')
  ('directement', 'ADV')
  ('donc', 'ADV')
  ('définitivement', 'ADV')
  ('déjà', 'ADV')
  ('désormais', 'ADV')
  ('encore', 'ADV')
  ('enfin', 'ADV')
  ('ensemble', 'ADV')
  ('ensuite', 'ADV')
  ('entièrement', 'ADV')
  ('environ', 'ADV')
  ('essentiellement', 'ADV')
  ('extrêmement', 'ADV')
  ('facilement', 'ADV')
  ('finalement', 'ADV')
  ('fort', 'ADV')
  ('fortement', 'ADV')
  ('généralement', 'ADV')
  ('hier', 'ADV')
  ('ici', 'ADV')
  ('immédiatement', 'ADV')
  ('initialement', 'ADV')
  ('jamais', 'ADV')
  ('juste', 'ADV

In [281]:
i = 17
    
    
print(f'Cluster {i}:')
for member in clusters[i]:
    print(f'  {unique_lemma[member]}')

Cluster 17:
  ('/', 'ADP')
  ('=', 'VERB')
  ('après', 'ADP')
  ('avant', 'ADP')
  ('avec', 'ADP')
  ('chez', 'ADP')
  ('comme', 'ADP')
  ('comme', 'SCONJ')
  ('concernant', 'ADP')
  ('contre', 'ADP')
  ('dans', 'ADP')
  ('de', 'ADP')
  ('depuis', 'ADP')
  ('derrière', 'ADP')
  ('devant', 'ADP')
  ('durant', 'ADP')
  ('dès', 'ADP')
  ('en', 'ADP')
  ('entre', 'ADP')
  ('envers', 'ADP')
  ('lorsque', 'SCONJ')
  ('malgré', 'ADP')
  ('outre', 'ADP')
  ('par', 'ADP')
  ('parmi', 'ADP')
  ('pendant', 'ADP')
  ('pour', 'ADP')
  ('puisque', 'SCONJ')
  ('quand', 'SCONJ')
  ('que', 'SCONJ')
  ('sans', 'ADP')
  ('sauf', 'ADP')
  ('selon', 'ADP')
  ('si', 'SCONJ')
  ('sous', 'ADP')
  ('suivant', 'ADP')
  ('sur', 'ADP')
  ('vers', 'ADP')
  ('via', 'ADP')
  ('à', 'ADP')


In [206]:
print(clusters[5])
print(unique_lemma[clusters[5][0]][1])

[147, 519, 620, 631, 654, 713, 727, 740, 773, 781, 785, 808, 809, 810, 812, 813, 819, 826, 829, 856, 1078, 1249, 1257, 1379, 1385, 1436, 1459, 1460, 1760, 1966, 1967, 1991, 2099, 2185, 2201, 2219, 2231, 2273, 2284, 2313, 2348, 2397, 2442, 2449, 2451, 2647, 2649, 2651, 2656, 2772, 2924, 2972, 3002, 3127, 3163, 3186, 3246, 3351, 3367, 3383]
X


In [261]:
pie_chart = {}
for cluster, members in clusters.items():
    pie_chart[cluster] = {}
    for member in members:
        if unique_lemma[member][1] in pie_chart[cluster]:
            pie_chart[cluster][unique_lemma[member][1]] += 1
        else:
            pie_chart[cluster][unique_lemma[member][1]] = 1
    

In [262]:
print(pie_chart)

{1: {'ADV': 108}, 2: {'ADV': 15, 'PRON': 6, 'ADP': 1}, 3: {'PRON': 22, 'INTJ': 1, 'CCONJ': 3, 'ADV': 3, 'ADJ': 1}, 4: {'NUM': 41, 'ADJ': 75, 'DET': 15}, 5: {'CCONJ': 11}, 6: {'PROPN': 13, 'X': 2, 'ADP': 1}, 7: {'PROPN': 73, 'NOUN': 3}, 8: {'PRON': 12}, 9: {'NOUN': 22, 'PROPN': 93, 'NUM': 1}, 10: {'NUM': 22, 'PROPN': 27, 'NOUN': 1362, 'ADJ': 1}, 11: {'NUM': 102}, 12: {'PROPN': 4, 'ADJ': 289}, 13: {'ADJ': 36, 'VERB': 69}, 14: {'NOUN': 13}, 15: {'X': 1, 'INTJ': 1}, 16: {'VERB': 486, 'AUX': 3}, 17: {'ADP': 33, 'VERB': 1, 'SCONJ': 6}}


In [302]:
# make pie chart with plotly
import plotly.express as px
import plotly.graph_objects as go

# Function to create pie chart for a given cluster
def create_pie_chart(cluster_number):
    labels = [f'{k} ({v})' for k, v in pie_chart[cluster_number].items()]
    values = list(pie_chart[cluster_number].values())
    fig = go.Figure(data=[go.Pie(labels=labels, values=values)])
    return fig

# Create initial pie chart
fig = create_pie_chart(1)

# Add dropdown menu
dropdown_buttons = [
    {
        'label': f'Cluster {i}',
        'method': 'update',
        'args': [{'values': [list(pie_chart[i].values())], 'labels': [[f'{k} ({v})' for k, v in pie_chart[i].items()]]}]
    } for i in pie_chart.keys()
]

fig.update_layout(
    updatemenus=[
        {
            'buttons': dropdown_buttons,
            'direction': 'down',
            'showactive': True,
        }
    ]
)

fig.show()

In [303]:
fig.write_html("French_all_lex_units_pie.html")

In [293]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Create a DataFrame for Plotly
df = pd.DataFrame({
    'PCA1': X_pca[:, 0],
    'PCA2': X_pca[:, 1],
    'Cluster': labels,
    'Word': unique_lemma
})
print(df)
# pca1_min, pca1_max = df['PCA1'].min(), df['PCA1'].max()
# pca2_min, pca2_max = df['PCA2'].min(), df['PCA2'].max()
# Create a scatter plot with Plotly
fig = go.Figure()

# Add traces for each cluster
for cluster in range(1, optimal_clusters + 1):
    cluster_data = df[df['Cluster'] == cluster]
    fig.add_trace(go.Scatter(
        x=cluster_data['PCA1'],
        y=cluster_data['PCA2'],
        mode='markers',
        marker=dict(size=10),
        name=f'Cluster {cluster}',
        text=cluster_data['Word'],
        # hoverinfo='text'
        hovertemplate='%{text}<extra></extra>',
    ))

# Update layout with dropdown menu
fig.update_layout(
    title='Word Clusters',
    xaxis_title='PCA1',
    yaxis_title='PCA2',
    # xaxis=dict(range=[pca1_min, pca1_max]),
    # yaxis=dict(range=[pca2_min, pca2_max]),
    updatemenus=[
        {
            'buttons': [
                {
                    'label': 'All Clusters',
                    'method': 'update',
                    'args': [{'visible': [True] * optimal_clusters},
                             {'title': 'All Clusters'}]
                }
            ] + [
                {
                    'label': f'Cluster {i}',
                    'method': 'update',
                    'args': [{'visible': [j == i - 1 for j in range(optimal_clusters)]},
                             {'title': f'Cluster {i}'}]
                } for i in range(1, optimal_clusters + 1)
            ],
            'direction': 'down',
            'showactive': True
        }
    ]
)

fig.show()

          PCA1      PCA2  Cluster            Word
0    -0.045762 -0.025284        9       ($, NOUN)
1    -0.003882  0.030481        9       (%, NOUN)
2    -0.042612 -0.027479        5      (&, CCONJ)
3    -0.036073 -0.012170       17        (/, ADP)
4    -0.022019 -0.023676        5      (/, CCONJ)
...        ...       ...      ...             ...
2973 -0.038777 -0.020583       10     (œil, NOUN)
2974 -0.041454 -0.022035       10     (œuf, NOUN)
2975 -0.018135 -0.003849       10   (œuvre, NOUN)
2976 -0.044354 -0.024935       16  (œuvrer, VERB)
2977 -0.040655 -0.016649        9       (€, NOUN)

[2978 rows x 4 columns]


In [297]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.manifold import TSNE
import plotly.express as px
import plotly.graph_objects as go

tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X)

In [300]:


# Create a DataFrame for Plotly
df = pd.DataFrame({
    'PCA1': X_tsne[:, 0],
    'PCA2': X_tsne[:, 1],
    'Cluster': labels,
    'Word': unique_lemma
})
# pca1_min, pca1_max = df['PCA1'].min(), df['PCA1'].max()
# pca2_min, pca2_max = df['PCA2'].min(), df['PCA2'].max()
# Create a scatter plot with Plotly
fig = go.Figure()

# Add traces for each cluster
for cluster in range(1, optimal_clusters + 1):
    cluster_data = df[df['Cluster'] == cluster]
    fig.add_trace(go.Scatter(
        x=cluster_data['PCA1'],
        y=cluster_data['PCA2'],
        mode='markers',
        marker=dict(size=10),
        name=f'Cluster {cluster}',
        text=cluster_data['Word'],
        hovertemplate='%{text}<extra></extra>',
    ))

# Update layout with dropdown menu
fig.update_layout(
    title='Word Clusters',
    xaxis_title='tsne1',
    yaxis_title='tsne2',
    width=1600,  # Set the width of the figure
    height=800,  # Set the height of the figure
    # xaxis=dict(range=[pca1_min, pca1_max]),
    # yaxis=dict(range=[pca2_min, pca2_max]),
    updatemenus=[
        {
            'buttons': [
                {
                    'label': 'All Clusters',
                    'method': 'update',
                    'args': [{'visible': [True] * optimal_clusters},
                             {'title': 'All Clusters'}]
                }
            ] + [
                {
                    'label': f'Cluster {i}',
                    'method': 'update',
                    'args': [{'visible': [j == i - 1 for j in range(optimal_clusters)]},
                             {'title': f'Cluster {i}'}]
                } for i in range(1, optimal_clusters + 1)
            ],
            'direction': 'down',
            'showactive': True
        }
    ]
)

fig.show()

In [301]:
fig.write_html("all_lex_units_tsne.html")

In [210]:
clusters_pos = {}
for cluster, members in clusters.items():
    clusters_pos[cluster] = []
    for member in members:
        clusters_pos[cluster].append(upos_of_matches[unique_lemma[member]])

KeyError: ('$', 'SYM')

In [ ]:
import plotly.graph_objs as go
from plotly.subplots import make_subplots

# Function to create a pie chart for a given cluster
def create_pie_chart(cluster_id):
    pos_counts = {}
    for pos_list in clusters_pos[cluster_id]:
        # Join the list into a single string to represent the combination
        pos_key = ','.join(sorted(pos_list))
        pos_counts[pos_key] = pos_counts.get(pos_key, 0) + 1

    labels = list(pos_counts.keys())
    values = list(pos_counts.values())

    return labels, values

# Create dropdown menu
fig = go.Figure()

# Add traces for each cluster
for cluster_id in clusters_pos.keys():
    labels, values = create_pie_chart(cluster_id)
    fig.add_trace(
        go.Pie(labels=labels, values=values, visible=(cluster_id == list(clusters.keys())[0]))
    )

# Update layout with dropdown menu
fig.update_layout(
    updatemenus=[
        {
            "buttons": [
                {
                    "label": f"Cluster {cluster_id}",
                    "method": "update",
                    "args": [
                        {"visible": [cluster_id == c for c in clusters_pos.keys()]},
                        {"title": f"POS Distribution for Cluster {cluster_id}"}
                    ]
                }
                for cluster_id in clusters_pos.keys()
            ],
            "direction": "down",
            "showactive": True,
        }
    ]
)

# Show the figure
fig.show()

In [ ]:
import plotly.graph_objs as go
from collections import defaultdict
# Function to create sunburst chart data for a given cluster
def create_sunburst_data(cluster_id):
    pos_hierarchy = defaultdict(lambda: defaultdict(int))

    for pos_list in clusters_pos[cluster_id]:
        if len(pos_list) == 1:
            pos_hierarchy[pos_list[0]][''] += 1
        else:
            combination_key = 'Combination POS'
            pos_hierarchy[combination_key][','.join(sorted(pos_list))] += 1

    labels = []
    parents = []
    values = []

    for pos, combinations in pos_hierarchy.items():
        labels.append(pos)
        parents.append('')
        values.append(sum(combinations.values()))
        for combination, count in combinations.items():
            if combination:  # Skip the empty string key for single POS
                labels.append(combination)
                parents.append(pos)
                values.append(count)

    return labels, parents, values

# Create the figure with dropdown menu
fig = go.Figure()

# Add traces for each cluster
for cluster_id in clusters_pos.keys():
    labels, parents, values = create_sunburst_data(cluster_id)
    fig.add_trace(
        go.Sunburst(
            labels=labels,
            parents=parents,
            values=values,
            branchvalues="total",
            hovertemplate='<b>%{label}</b><br>Percentage: %{percentRoot:.2f}%<extra></extra><br>Count: %{value}',
            visible=(cluster_id == list(clusters_pos.keys())[0])
        )
    )

# Update layout with dropdown menu
fig.update_layout(
    updatemenus=[
        {
            "buttons": [
                {
                    "label": f"Cluster {cluster_id}",
                    "method": "update",
                    "args": [
                        {"visible": [cluster_id == c for c in clusters_pos.keys()]},
                        {"title": f"POS Distribution for Cluster {cluster_id}"}
                    ]
                }
                for cluster_id in clusters_pos.keys()
            ],
            "direction": "down",
            "showactive": True,
        }
    ],
    title_text="POS Distribution for Cluster 1",
    width = 800,
    height = 800
)

# Show the figure
fig.show()

In [ ]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# Compute the silhouette score
sil_score = silhouette_score(X, labels, metric='cosine')
print(f'Silhouette Score: {sil_score}')

# Compute the Calinski-Harabasz Index
calinski_harabasz = calinski_harabasz_score(X, labels)
print(f'Calinski-Harabasz Index: {calinski_harabasz}')

# Compute the Davies-Bouldin Index
davies_bouldin = davies_bouldin_score(X, labels)
print(f'Davies-Bouldin Index: {davies_bouldin}')

# Dunn index (assuming you have a function `dunn` defined elsewhere)
# clusters_d = [X[labels == j] for j in np.unique(labels)]
# dunn_index = dunn(clusters_d)
# print(f'Dunn Index: {dunn_index}')

Silhouette Score: 0.16885741561705425
Calinski-Harabasz Index: 11.481160970840381
Davies-Bouldin Index: 2.3386502810135417


In [ ]:
import plotly.graph_objs as go
from collections import defaultdict
# Function to create sunburst chart data for a given cluster
def create_sunburst_data(cluster_id):
    pos_hierarchy = defaultdict(lambda: defaultdict(int))

    for pos_list in clusters_pos[cluster_id]:
        if len(pos_list) == 1:
            pos_hierarchy[pos_list[0]][''] += 1
        else:
            combination_key = 'Combination POS'
            pos_hierarchy[combination_key][','.join(sorted(pos_list))] += 1

    labels = []
    parents = []
    values = []

    for pos, combinations in pos_hierarchy.items():
        labels.append(pos)
        parents.append('')
        values.append(sum(combinations.values()))
        for combination, count in combinations.items():
            if combination:  # Skip the empty string key for single POS
                labels.append(combination)
                parents.append(pos)
                values.append(count)

    return labels, parents, values

def get_words_for_cluster(cluster_id):
    words = [unique_lemma[member] for member in clusters[cluster_id]]
    return '<br>'.join(words)
# Create the figure with dropdown menu
fig = go.Figure()

# Add traces for each cluster
for cluster_id in clusters_pos.keys():
    labels, parents, values = create_sunburst_data(cluster_id)
    fig.add_trace(
        go.Sunburst(
            labels=labels,
            parents=parents,
            values=values,
            branchvalues="total",
            hovertemplate='<b>%{label}</b><br>Percentage: %{percentRoot:.2f}%<extra></extra><br>Count: %{value}',
            visible=(cluster_id == list(clusters_pos.keys())[0])
        )
    )

# Add annotations for words
for cluster_id in clusters.keys():
    words_text = get_words_for_cluster(cluster_id)
    fig.add_annotation(
        text=f"Words in Cluster {cluster_id}:<br>{words_text}",
        showarrow=False,
        xref="paper", yref="paper",
        x=1.1, y=0.8,
        align="left",
        visible=(cluster_id == list(clusters.keys())[0])
    )

# Update layout with dropdown menu
fig.update_layout(
    updatemenus=[
        {
            "buttons": [
                {
                    "label": f"Cluster {cluster_id}",
                    "method": "update",
                    "args": [
                        {"visible": [c == cluster_id for c in clusters.keys()] * 2},  # Update both traces and annotations
                        {"title": f"POS Distribution for Cluster {cluster_id}"}
                    ]
                }
                for cluster_id in clusters.keys()
            ],
            "direction": "down",
            "showactive": True,
            "x": 1.1,
            "xanchor": "left",
            "y": 1.1,
            "yanchor": "top"
        }
    ],
    title_text="POS Distribution for Cluster 1",  # Default title
    width=1000,  # Set the width of the chart
    height=800  # Set the height of the chart
)

# Show the figure
fig.show()

In [ ]:
import dash
from dash import dcc, html, Input, Output
import plotly.graph_objs as go
from collections import defaultdict

# Function to create sunburst chart data for a given cluster
def create_sunburst_data(cluster_id):
    pos_hierarchy = defaultdict(lambda: defaultdict(int))

    for pos_list in clusters_pos[cluster_id]:
        if len(pos_list) == 1:
            pos_hierarchy[pos_list[0]][''] += 1
        else:
            combination_key = 'Combination POS'
            pos_hierarchy[combination_key][','.join(sorted(pos_list))] += 1

    labels = []
    parents = []
    values = []

    for pos, combinations in pos_hierarchy.items():
        labels.append(pos)
        parents.append('')
        values.append(sum(combinations.values()))
        for combination, count in combinations.items():
            if combination:  # Skip the empty string key for single POS
                labels.append(combination)
                parents.append(pos)
                values.append(count)

    return labels, parents, values

# Initialize the Dash app
app = dash.Dash(__name__)

app.layout = html.Div([
    dcc.Graph(id='sunburst-chart'),
    html.Div(id='word-list')
])

# Callback to update the sunburst chart based on the selected cluster
@app.callback(
    Output('sunburst-chart', 'figure'),
    Input('sunburst-chart', 'clickData')
)
def update_sunburst(clickData):
    cluster_id = 1  # Default cluster
    if clickData:
        cluster_id = clickData['points'][0]['label']

    labels, parents, values = create_sunburst_data(cluster_id)
    fig = go.Figure(go.Sunburst(
        labels=labels,
        parents=parents,
        values=values,
        branchvalues="total",
        hovertemplate='<b>%{label}</b><br>Percentage: %{percentRoot:.2f}%<extra></extra>'
    ))
    return fig

# Callback to update the word list based on the selected slice
@app.callback(
    Output('word-list', 'children'),
    Input('sunburst-chart', 'clickData')
)
def display_words(clickData):
    if not clickData:
        return "Click on a slice to see the words."

    selected_label = clickData['points'][0]['label']
    for member in clusters_pos[1]:
        if selected_label in member:
            words = [unique_lemma[member] for member in clusters[1]]
    return html.Div([
        html.H5(f"Words in {selected_label}:"),
        html.Ul([html.Li(word) for word in words])
    ])

# Run the app
if __name__ == '__main__':
    app.run_server(debug=True)


In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Create a DataFrame for Plotly
df = pd.DataFrame({
    'PCA1': X_pca[:, 0],
    'PCA2': X_pca[:, 1],
    'Cluster': labels,
    'Word': unique_lemma
})
# pca1_min, pca1_max = df['PCA1'].min(), df['PCA1'].max()
# pca2_min, pca2_max = df['PCA2'].min(), df['PCA2'].max()
# Create a scatter plot with Plotly
fig = go.Figure()

# Add traces for each cluster
for cluster in range(1, optimal_clusters + 1):
    cluster_data = df[df['Cluster'] == cluster]
    fig.add_trace(go.Scatter(
        x=cluster_data['PCA1'],
        y=cluster_data['PCA2'],
        mode='markers',
        marker=dict(size=10),
        name=f'Cluster {cluster}',
        text=cluster_data['Word'],
        hoverinfo='text'
    ))

# Update layout with dropdown menu
fig.update_layout(
    title='Word Clusters',
    xaxis_title='PCA1',
    yaxis_title='PCA2',
    # xaxis=dict(range=[pca1_min, pca1_max]),
    # yaxis=dict(range=[pca2_min, pca2_max]),
    updatemenus=[
        {
            'buttons': [
                {
                    'label': 'All Clusters',
                    'method': 'update',
                    'args': [{'visible': [True] * optimal_clusters},
                             {'title': 'All Clusters'}]
                }
            ] + [
                {
                    'label': f'Cluster {i}',
                    'method': 'update',
                    'args': [{'visible': [j == i - 1 for j in range(optimal_clusters)]},
                             {'title': f'Cluster {i}'}]
                } for i in range(1, optimal_clusters + 1)
            ],
            'direction': 'down',
            'showactive': True
        }
    ]
)

fig.show()

In [ ]:
import plotly.express as px
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import cosine_distances
import pandas as pd

# Compute the cosine distance matrix
distance_matrix = cosine_distances(X)

# Fit DBSCAN model
db = DBSCAN(eps=0.09, min_samples=5, metric='precomputed')
labels = db.fit_predict(distance_matrix)

#reduce dimensionality for visualization
from sklearn.decomposition import PCA
# Reduce dimensionality to 2D using PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Create a DataFrame for Plotly
df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df['Label'] = labels
df['Lemma'] = unique_lemma  # Assuming unique_adv is a list of adverb names

# Convert labels to categorical type
df['Label'] = df['Label'].astype(str)

# Create a scatter plot with Plotly
fig = px.scatter(df, x='PC1', y='PC2', color='Label', hover_data=['Lemma'],
                 color_discrete_sequence=px.colors.qualitative.Plotly)

fig.update_layout(
    title='DBSCAN Clustering',
    xaxis_title='Principal Component 1',
    yaxis_title='Principal Component 2',
    font=dict(size=12)
)

fig.show()

In [ ]:
# see what there is in each cluster

db = DBSCAN(eps=0.1, min_samples=5, metric='precomputed')
labels = db.fit_predict(distance_matrix)
# Create a dictionary to store clusters
clusters = {}
for i, label in enumerate(labels):
    if label not in clusters:
        clusters[label] = []
    clusters[label].append(i)

# Print the contents of each cluster
for cluster, members in clusters.items():
    print(f'Cluster {cluster}:')
    for member in members:
        print(f'  {unique_lemma[member]}')

Cluster -1:
  $
  %
  &
  +
  /
  0
  1
  =
  A
  AS
  Alex
  Anne
  Awards
  B
  Black
  C
  City
  Club
  Company
  Côte
  Disney
  Hall
  Hugues
  International
  Internet
  Jeanne
  Johnson
  K
  Lord
  M.
  Man
  Maria
  Me
  Milan
  Napoléon
  National
  Notre-Dame
  PS
  Peter
  Pokémon
  Polisario
  Ray
  Sarkozy
  Serbie
  TV
  The
  US
  United
  Windows
  World
  X
  __undefined__
  a
  absolument
  afin
  agir
  ailleurs
  ainsi
  alors
  amoureux
  armer
  arrêté
  attention
  aucun
  auparavant
  autant
  avant
  avoir
  avérer
  beaucoup
  bien
  bref
  c'est-à-dire
  capable
  car
  cause
  ce
  ceci
  cela
  celui
  celui-ci
  censé
  certain
  chacun
  chaque
  comment
  comparer
  connu
  conseiller
  contraire
  contrairement
  coûteux
  d'
  davantage
  dehors
  del
  demi
  di
  différent
  discuter
  divers
  dollar
  dommage
  dont
  du
  décéder
  délicieux
  désirer
  endémique
  enfermer
  environ
  et
  etc.
  eux
  eux-mêmes
  ex
  face
  facile
  fait
  fa